<a href="https://colab.research.google.com/github/vaisshnavee1410/Recommendation_System.ipynb/blob/main/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **RECOMMENDATION SYSTEM**

## **DATA DESCRIPTION:**


Unique ID of each anime.

Anime title.

Anime broadcast type, such as TV, OVA, etc.

Anime genre.

The number of episodes of each anime.

The average rating for each anime compared to the number of users who gave ratings.

Number of community members for each anime.

### **OBJECTIVE:**

The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset.

### **DATASET:**

Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

### **DATAPREPROCESSING:**

Load the dataset into a suitable data structure (e.g., pandas DataFrame).

Handle missing values, if any.

Explore the dataset to understand its structure and attributes.


In [35]:
#load the dataset
import pandas as pd
df = pd.read_csv('anime.csv')

# Check for missing values
missing_values = df.isnull().sum()

# Display missing values
print("Missing values per column:")
print(missing_values)

# Drop rows with missing values (if needed)
df_cleaned = df.dropna()

# Summary statistics
print("\nSummary statistics:")
print(df_cleaned.describe(include='all'))

# Display basic info about the dataset
print("Dataset Info:")
df_cleaned.info()

# Display column data types
print("\nData Types:")
print(df_cleaned.dtypes)

# Check unique values in categorical columns
print("\nUnique values in 'type':")
print(df_cleaned['type'].value_counts())

print("\nTop 10 most common genres:")
from collections import Counter
import itertools

# Split and count genres
genre_split = df_cleaned['genre'].str.split(', ')
all_genres = list(itertools.chain.from_iterable(genre_split))
genre_counts = Counter(all_genres)
print(genre_counts.most_common(10))

# Check basic statistics for numeric columns
print("\nSummary statistics for numeric columns:")
print(df_cleaned[['rating', 'members']].describe())

# Correlation between numeric attributes
print("\nCorrelation matrix:")
print(df_cleaned[['rating', 'members']].corr())

Missing values per column:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Summary statistics:
            anime_id              name   genre   type episodes        rating  \
count   12017.000000             12017   12017  12017    12017  12017.000000   
unique           NaN             12015    3229      6      187           NaN   
top              NaN  Saru Kani Gassen  Hentai     TV        1           NaN   
freq             NaN                 2     816   3668     5571           NaN   
mean    13638.001165               NaN     NaN    NaN      NaN      6.478264   
std     11231.076675               NaN     NaN    NaN      NaN      1.023857   
min         1.000000               NaN     NaN    NaN      NaN      1.670000   
25%      3391.000000               NaN     NaN    NaN      NaN      5.890000   
50%      9959.000000               NaN     NaN    NaN      NaN      6.570000   
75%     23729.000000       

### **FEATURE EXTRACTION:**

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.


In [36]:
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Copy the cleaned dataset
df_features = df_cleaned.copy()

# Parse genres into list
df_features['genre'] = df_features['genre'].apply(lambda x: x.split(', '))

# Multi-hot encoding for genres
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df_features['genre'])
genres_df = pd.DataFrame(genres_encoded, columns=mlb.classes_)

# One-hot encoding for 'type'
type_encoded = pd.get_dummies(df_features['type'], prefix='type')

# Normalize 'rating' and 'members'
scaler = MinMaxScaler()
numerical_scaled = scaler.fit_transform(df_features[['rating', 'members']])
numerical_df = pd.DataFrame(numerical_scaled, columns=['rating_scaled', 'members_scaled'])

# Combine all features into one DataFrame
feature_matrix = pd.concat([genres_df, type_encoded, numerical_df], axis=1)

# Display shape and first few rows
print("Feature matrix shape:", feature_matrix.shape)
print(feature_matrix.head())

Feature matrix shape: (12284, 51)
   Action  Adventure  Cars  Comedy  Dementia  Demons  Drama  Ecchi  Fantasy  \
0     0.0        0.0   0.0     0.0       0.0     0.0    1.0    0.0      0.0   
1     1.0        1.0   0.0     0.0       0.0     0.0    1.0    0.0      1.0   
2     1.0        0.0   0.0     1.0       0.0     0.0    0.0    0.0      0.0   
3     0.0        0.0   0.0     0.0       0.0     0.0    0.0    0.0      0.0   
4     1.0        0.0   0.0     1.0       0.0     0.0    0.0    0.0      0.0   

   Game  ...  Yaoi  Yuri  type_Movie  type_Music  type_ONA  type_OVA  \
0   0.0  ...   0.0   0.0        True       False     False     False   
1   0.0  ...   0.0   0.0       False       False     False     False   
2   0.0  ...   0.0   0.0       False       False     False     False   
3   0.0  ...   0.0   0.0       False       False     False     False   
4   0.0  ...   0.0   0.0       False       False     False     False   

   type_Special  type_TV  rating_scaled  members_scaled  


### **RECOMMENDATION SYSTEM:**

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.

In [37]:
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Copy the cleaned dataset
df_features = df_cleaned.copy()

# Parse genres into list
df_features['genre'] = df_features['genre'].apply(lambda x: x.split(', '))

# Multi-hot encoding for genres
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df_features['genre'])
genres_df = pd.DataFrame(genres_encoded, columns=mlb.classes_)

# One-hot encoding for 'type'
type_encoded = pd.get_dummies(df_features['type'], prefix='type')

# Normalize 'rating' and 'members'
scaler = MinMaxScaler()

# Before scaling, impute NaN values with the mean
# This will replace missing values with the average value for that column
# You can use different imputation strategies, such as median, mode, or others
df_features[['rating', 'members']] = df_features[['rating', 'members']].fillna(df_features[['rating', 'members']].mean())

numerical_scaled = scaler.fit_transform(df_features[['rating', 'members']])
numerical_df = pd.DataFrame(numerical_scaled, columns=['rating_scaled', 'members_scaled'])

# Combine all features into one DataFrame
feature_matrix = pd.concat([genres_df, type_encoded, numerical_df], axis=1)

# Display shape and first few rows
print("Feature matrix shape:", feature_matrix.shape)
print(feature_matrix.head())

Feature matrix shape: (12284, 51)
   Action  Adventure  Cars  Comedy  Dementia  Demons  Drama  Ecchi  Fantasy  \
0     0.0        0.0   0.0     0.0       0.0     0.0    1.0    0.0      0.0   
1     1.0        1.0   0.0     0.0       0.0     0.0    1.0    0.0      1.0   
2     1.0        0.0   0.0     1.0       0.0     0.0    0.0    0.0      0.0   
3     0.0        0.0   0.0     0.0       0.0     0.0    0.0    0.0      0.0   
4     1.0        0.0   0.0     1.0       0.0     0.0    0.0    0.0      0.0   

   Game  ...  Yaoi  Yuri  type_Movie  type_Music  type_ONA  type_OVA  \
0   0.0  ...   0.0   0.0        True       False     False     False   
1   0.0  ...   0.0   0.0       False       False     False     False   
2   0.0  ...   0.0   0.0       False       False     False     False   
3   0.0  ...   0.0   0.0       False       False     False     False   
4   0.0  ...   0.0   0.0       False       False     False     False   

   type_Special  type_TV  rating_scaled  members_scaled  


### **EVALUATION:**

Split the dataset into training and testing sets.
Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.
Analyze the performance of the recommendation system and identify areas of improvement.


In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from scipy.sparse import csr_matrix

# Step 1: Load the dataset
df = pd.read_csv("anime.csv")


# Check the actual column names in your DataFrame
print(df.columns)

# Step 6: Evaluate using test data
y_true = []
y_pred = []

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000


### **INTERVIEW QUESTIONS:**

* **Can you explain the difference between user-based and item-based collaborative filtering?**

**1.User-Based Collaborative Filtering:**

**Concept:**

This method recommends items to a user based on the preferences of similar users.

**How it works:**

•	Find users who are similar to the target user (e.g., they rated the same items similarly).

•	Recommend items that those similar users liked but the target user hasn’t interacted with yet.

**Example:**

If User A and User B both like Naruto and One Piece, and User B also likes Bleach, then Bleach might be recommended to User A.

**Pros:**

	•	Personalized and intuitive.
	•	Works well when user preferences are stable and similar patterns exist.

**Cons:**

	•	Doesn’t scale well with a large number of users.
	•	Suffers when users have little overlap (sparsity problem).

**2. Item-Based Collaborative Filtering:**

**Concept:**

This method recommends items that are similar to items the user already liked.

**How it works:**

•	Find items that are similar to the ones the user rated highly.

•	Recommend those similar items.

**Example:**

If a user liked Naruto, and Naruto is similar to Bleach (because many users liked both), then Bleach will be recommended.

**Pros:**

	•	More stable than user-based filtering since item relationships don’t change frequently.
	•	Better suited for large user bases and small item catalogs.

**Cons:**

	•	Can miss recommendations if item-item similarities aren’t informative.
	•	Assumes that item similarity is enough to predict user preference.

**3.DIFFERENCE:**

User-based and item-based collaborative filtering are both techniques used in recommendation systems, but they differ in focus and approach. User-based collaborative filtering recommends items to a user based on the preferences of other users who are similar to them. It assumes that if two users have liked similar items in the past, they are likely to enjoy similar things in the future. In contrast, item-based collaborative filtering recommends items that are similar to those the user has already liked, regardless of what other users have done. It operates on the principle that items co-liked by many users are likely to be similar, so if a user enjoyed one item, they might enjoy another similar one. While user-based filtering is more personalized, it can struggle with scalability and data sparsity as the number of users grows. Item-based filtering tends to be more stable and scalable, especially when the item catalog is smaller than the user base, but it relies heavily on having meaningful similarities between items.

* **What is collaborative filtering, and how does it work?**

Collaborative filtering uses patterns in user-item interactions (such as ratings or clicks) to make predictions. It doesn’t require any knowledge about the items themselves (like genres or descriptions), only how users interact with them.


**Key Steps in Collaborative Filtering:**

1.	Build a user-item matrix (users as rows, items as columns, values as ratings or interactions).

2.	Calculate similarities between users or items (commonly using cosine similarity or Pearson correlation).

3.	Generate recommendations by identifying items with high similarity scores.

**Advantages:**

	•	Doesn’t require item metadata or content.
	•	Can capture complex patterns in user behavior.

**Disadvantages:**

	•	Suffers from the cold start problem (difficulty recommending for new users/items).
	•	Can be affected by data sparsity (most users rate only a few items).
	•	Requires a large volume of interaction data to be effective.
